In [1]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
ROOT = Path.cwd()
INPUT_CSV = str(ROOT /"orders.csv")
OUTPUT_DIR = str(ROOT / "output" / "week3_delayed_by_supplier")

In [3]:
spark = (
        SparkSession.builder.appName("SupplyChainWeek3")
        .master("local[*]")
        .config("spark.ui.showConsoleProgress", "false")
        .getOrCreate()
    )
spark.sparkContext.setLogLevel("ERROR")

In [4]:
 # 1. Load
df = spark.read.csv(INPUT_CSV, header=True, inferSchema=True)
print(f"Loaded {df.count()} rows")

Loaded 201 rows


In [6]:
# 2. Clean + compute delay
df = (
        df.dropDuplicates(["order_id"])
        # .dropna(subset=["supplier_id", "delivery_date"])
        .withColumn("expected_delivery_date", F.to_date("expected_delivery_date"))
        .withColumn("delivery_date", F.to_date("delivery_date"))
        .withColumn("delay_days", F.datediff("delivery_date", "expected_delivery_date"))
    )

In [7]:
 # 3. Filter delayed shipments
delayed = df.filter(F.col("delay_days") > 0)
print(f"Delayed shipments: {delayed.count()}")

Delayed shipments: 125


In [8]:
result = (
        delayed.groupBy("supplier_id")
        .agg(
            F.count("*").alias("delayed_orders"),
            F.round(F.avg("delay_days"), 2).alias("avg_delay_days"),
        )
        .orderBy(F.desc("delayed_orders"))
    )
result.show()

+-----------+--------------+--------------+
|supplier_id|delayed_orders|avg_delay_days|
+-----------+--------------+--------------+
|       S002|            21|          3.86|
|       S005|            20|           4.1|
|       S006|            18|          5.11|
|       S004|            16|          4.69|
|       S008|            14|          3.86|
|       S003|            14|          4.14|
|       S001|            12|          4.33|
|       S007|            10|           4.3|
+-----------+--------------+--------------+



In [9]:
# 5. Save (single CSV part file + Parquet)
result.coalesce(1).write.mode("overwrite").option("header", True).csv(OUTPUT_DIR + "_csv")
result.write.mode("overwrite").parquet(OUTPUT_DIR + "_parquet")
print(f"Saved results under {OUTPUT_DIR}_csv and {OUTPUT_DIR}_parquet")

spark.stop()

Saved results under /content/output/week3_delayed_by_supplier_csv and /content/output/week3_delayed_by_supplier_parquet


In [10]:
import pandas as pd
pdf = pd.read_parquet(OUTPUT_DIR + "_parquet")
display(pdf)

,supplier_id,delayed_orders,avg_delay_days
0,S002,21,3.86
1,S005,20,4.10
2,S006,18,5.11
3,S004,16,4.69
4,S008,14,3.86
5,S003,14,4.14
6,S001,12,4.33
7,S007,10,4.30
